In [ ]:
import os, subprocess
os.makedirs("build", exist_ok=True)

In [ ]:
# --- WaveDrom / VCD rendering utilities (auto-generated — do not edit) ---
import json, re, subprocess
from IPython.display import HTML, display

def _vcd_to_wavedrom(vcd_path, max_cycles=80, signals=None):
    """
    Minimal VCD → WaveDrom JSON converter.
    Works for single-bit and multi-bit signals from iverilog output.
    """
    with open(vcd_path) as f:
        vcd = f.read()

    # Parse variable declarations
    var_map = {}  # id_code → (name, width)
    for m in re.finditer(
        r"\$var\s+\w+\s+(\d+)\s+(\S+)\s+(\S+)", vcd
    ):
        width, code, name = int(m.group(1)), m.group(2), m.group(3)
        if signals is None or name in signals:
            var_map[code] = (name, width)

    if not var_map:
        print(f"⚠  No matching signals in {vcd_path}")
        return None

    # Parse value changes
    changes = {}  # id_code → [(time, value), ...]
    current_time = 0
    for line in vcd.splitlines():
        line = line.strip()
        if line.startswith("#"):
            current_time = int(line[1:])
        elif line.startswith("b"):
            parts = line.split()
            if len(parts) == 2 and parts[1] in var_map:
                val = int(parts[0][1:], 2) if parts[0][1:] else 0
                changes.setdefault(parts[1], []).append((current_time, val))
        elif len(line) >= 2 and line[0] in "01xXzZ" and line[1:] in var_map:
            code = line[1:]
            val = {"0": 0, "1": 1, "x": "x", "X": "x", "z": "z", "Z": "z"}[line[0]]
            changes.setdefault(code, []).append((current_time, val))

    # Determine time step (smallest non-zero delta)
    all_times = sorted({t for ch in changes.values() for t, _ in ch})
    if len(all_times) < 2:
        return None
    deltas = [all_times[i+1] - all_times[i] for i in range(len(all_times)-1) if all_times[i+1] != all_times[i]]
    if not deltas:
        return None
    step = min(deltas)
    end_time = min(all_times[-1], step * max_cycles) if max_cycles else all_times[-1]
    sample_times = list(range(0, end_time + 1, step))[:max_cycles]

    # Build WaveDrom signal list
    wave_signals = []
    for code, (name, width) in sorted(var_map.items(), key=lambda x: x[1][0]):
        ch = sorted(changes.get(code, []))
        # Sample at each time point
        samples = []
        cur_val = 0
        ci = 0
        for t in sample_times:
            while ci < len(ch) and ch[ci][0] <= t:
                cur_val = ch[ci][1]
                ci += 1
            samples.append(cur_val)

        if width == 1:
            # Single-bit: WaveDrom wave string
            wave_str = ""
            for v in samples:
                if v == "x":
                    wave_str += "x"
                elif v == "z":
                    wave_str += "z"
                else:
                    wave_str += str(int(v))
            wave_signals.append({"name": name, "wave": wave_str})
        else:
            # Multi-bit: use '=' for data with labels
            wave_str = ""
            data = []
            prev = None
            for v in samples:
                if v == prev:
                    wave_str += "."
                else:
                    wave_str += "="
                    data.append(f"0x{v:X}" if isinstance(v, int) else str(v))
                    prev = v
            wave_signals.append({"name": name, "wave": wave_str, "data": data})

    return {"signal": wave_signals, "config": {"hscale": 1}}


def show_waves(vcd_path="dump.vcd", max_cycles=80, signals=None, width=900):
    """Render VCD waveforms inline using WaveDrom."""
    wd = _vcd_to_wavedrom(vcd_path, max_cycles=max_cycles, signals=signals)
    if wd is None:
        print("No waveform data to display.")
        return
    wd_json = json.dumps(wd)
    html = f"""
    <div id="wd_{id(wd):x}"></div>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/wavedrom.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/skins/default.js"></script>
    <script>
    (function() {{
        var container = document.getElementById("wd_{id(wd):x}");
        container.innerHTML = '<div id="wd_tgt_{id(wd):x}"><script type="WaveDrom">' +
            JSON.stringify({wd_json}) + '</' + 'script></div>';
        WaveDrom.ProcessAll();
    }})();
    </script>
    """
    display(HTML(html))


def show_wavedrom(wave_dict, width=900):
    """Render a raw WaveDrom JSON dict inline."""
    wd_json = json.dumps(wave_dict)
    html = f"""
    <div id="wd_{id(wave_dict):x}"></div>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/wavedrom.min.js"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/wavedrom/3.5.0/skins/default.js"></script>
    <script>
    (function() {{
        var container = document.getElementById("wd_{id(wave_dict):x}");
        container.innerHTML = '<div id="wd_tgt_{id(wave_dict):x}"><script type="WaveDrom">' +
            JSON.stringify({wd_json}) + '</' + 'script></div>';
        WaveDrom.ProcessAll();
    }})();
    </script>
    """
    display(HTML(html))

print("✓ WaveDrom helpers loaded — use show_waves('dump.vcd') after simulation")

# Day 6 Lab: Testbenches & Simulation-Driven Development

## Course: Accelerated HDL for Digital System Design — Week 2, Session 6

---

## Objectives

By the end of this lab, you will:
- Write a comprehensive self-checking testbench with automated pass/fail reporting
- Use `task` blocks to organize test stimulus and checking
- Apply file-driven testing with `$readmemh`
- Demonstrate exhaustive testing of a small combinational module

## Prerequisites

- Completed Days 1–5 labs (ALU from Day 3, debounce from Day 5)
- Watched Day 6 pre-class video (~50 min): testbench anatomy, self-checking patterns
- Copy your `alu_4bit.v` from Day 3 (or use the provided version)

## Key Rule: The Verification Contract

**From today forward, every lab deliverable must include:**
1. The design module(s)
2. A self-checking testbench with pass/fail output
3. A passing test report (terminal output screenshot)
4. Only after tests pass: the hardware demo

## Deliverables

| # | Exercise | Time | What to Submit |
|---|----------|------|----------------|
| 1 | ALU Testbench | 35 min | `tb_alu_4bit.v` — all tests pass + bug injection demo |
| 2 | Debounce Testbench | 25 min | `tb_debounce_thorough.v` — 4 scenarios pass |
| 3 | Counter Testbench | 20 min | `tb_counter.v` — rollover, reset, enable verified |
| 4 | File-Driven TB (stretch) | 15 min | `tb_hex_file.v` + `hex_vectors.hex` |
| 5 | Exhaustive Test (stretch) | 20 min | 1024 vectors, 0 failures |

**Primary deliverable:** Self-checking ALU testbench with automated pass/fail report.

---

## Exercise 1: Self-Checking ALU Testbench (35 min)

**SLOs: 6.1, 6.2, 6.3, 6.6**

### Part A: Create `tb_alu_4bit.v`

Use the starter file. Implement the `check_alu` task and complete the test vectors.

Requirements:
1. Clock generation (practice the pattern for sequential designs)
2. Waveform dump (`$dumpfile`/`$dumpvars`)
3. `check_alu` task: apply inputs, wait, compare result/carry/zero, report pass/fail
4. Final summary with total pass/fail counts

### Part B: Run and verify all tests pass

In [ ]:
!make sim TB=tb_alu_4bit.v SRCS="alu_4bit.v"

### Part C: Intentional bug injection

Change SUB to `a + b` in the ALU. Re-run the testbench. Confirm SUB tests fail.

---

#### 📝 Exercise 1 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile alu_4bit.v
// =============================================================================
// alu_4bit.v — 4-bit ALU (from Day 3, provided for testbench exercises)
// =============================================================================

module alu_4bit (
    input  wire [3:0] i_a,
    input  wire [3:0] i_b,
    input  wire [1:0] i_opcode,
    output reg  [3:0] o_result,
    output reg        o_carry,
    output wire       o_zero
);

    reg [4:0] r_temp;  // 5 bits for carry detection

    always @(*) begin
        r_temp = 5'd0;
        case (i_opcode)
            2'b00: r_temp = {1'b0, i_a} + {1'b0, i_b};  // ADD
            2'b01: r_temp = {1'b0, i_a} - {1'b0, i_b};  // SUB
            2'b10: r_temp = {1'b0, i_a & i_b};           // AND
            2'b11: r_temp = {1'b0, i_a | i_b};           // OR
        endcase
        o_result = r_temp[3:0];
        o_carry  = r_temp[4];
    end

    assign o_zero = (o_result == 4'd0);

endmodule

In [ ]:
%%writefile tb_alu_4bit.v
// =============================================================================
// tb_alu_4bit.v — Self-Checking ALU Testbench (Starter)
// Day 6, Exercise 1
// =============================================================================

`timescale 1ns / 1ps

module tb_alu_4bit;

    reg  [3:0] a, b;
    reg  [1:0] opcode;
    wire [3:0] result;
    wire       carry, zero;

    alu_4bit uut (
        .i_a(a), .i_b(b), .i_opcode(opcode),
        .o_result(result), .o_carry(carry), .o_zero(zero)
    );

    integer test_count = 0;
    integer fail_count = 0;

    // ---- TODO: Implement check_alu task ----
    // task check_alu;
    //   input [3:0] t_a, t_b;
    //   input [1:0] t_op;
    //   input [3:0] exp_result;
    //   input       exp_carry, exp_zero;
    // begin
    //   a = t_a; b = t_b; opcode = t_op;
    //   #10;
    //   test_count = test_count + 1;
    //   if (result !== exp_result || carry !== exp_carry || zero !== exp_zero) begin
    //     fail_count = fail_count + 1;
    //     $display("FAIL: op=%b a=%h b=%h | expected r=%h c=%b z=%b | got r=%h c=%b z=%b",
    //              t_op, t_a, t_b, exp_result, exp_carry, exp_zero,
    //              result, carry, zero);
    //   end else begin
    //     $display("PASS: op=%b a=%h b=%h = %h (c=%b z=%b)",
    //              t_op, t_a, t_b, result, carry, zero);
    //   end
    // end
    // endtask


    initial begin
        $dumpfile("dump.vcd");
        $dumpvars(0, tb_alu_4bit);

        $display("=== ALU 4-bit Self-Checking Testbench ===");
        $display("");

        // ---- TODO: ADD tests (opcode = 2'b00) ----
        // check_alu(4'h0, 4'h0, 2'b00, 4'h0, 1'b0, 1'b1);  // 0+0=0, z=1
        // check_alu(4'h3, 4'h5, 2'b00, 4'h8, 1'b0, 1'b0);  // 3+5=8
        // check_alu(4'hF, 4'h1, 2'b00, 4'h0, 1'b1, 1'b1);  // F+1=0, c=1, z=1
        // check_alu(4'h8, 4'h8, 2'b00, 4'h0, 1'b1, 1'b1);  // 8+8=0, c=1, z=1
        // check_alu(4'h7, 4'h8, 2'b00, 4'hF, 1'b0, 1'b0);  // 7+8=F

        // ---- TODO: SUB tests (opcode = 2'b01) ----

        // ---- TODO: AND tests (opcode = 2'b10) ----

        // ---- TODO: OR tests (opcode = 2'b11) ----


        // ---- Summary ----
        $display("");
        $display("========================================");
        $display("  %0d / %0d tests PASSED", test_count - fail_count, test_count);
        if (fail_count > 0)
            $display("  *** %0d FAILURES ***", fail_count);
        else
            $display("  All tests passed!");
        $display("========================================");
        $finish;
    end

endmodule

In [ ]:
%%writefile Makefile
# Auto-generated Makefile
TOP      = alu_4bit
SRCS     = alu_4bit.v
TB       = tb_alu_4bit.v
PCF      = ../../go_board.pcf
DEVICE   = hx1k
PACKAGE  = vq100
IVFLAGS  = -g2012 -Wall

sim: $(TB) $(SRCS)
	iverilog $(IVFLAGS) -o sim.vvp $(TB) $(SRCS)
	vvp sim.vvp

wave: sim
	gtkwave *.vcd &

synth: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP) -json $(TOP).json" $(SRCS)

prog: synth
	nextpnr-ice40 --$(DEVICE) --package $(PACKAGE) --pcf $(PCF) --json $(TOP).json --asc $(TOP).asc --freq 25
	icepack $(TOP).asc $(TOP).bin
	iceprog $(TOP).bin

stat: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP); stat" $(SRCS)

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: sim wave synth prog stat clean

In [ ]:
# Compile and simulate
!iverilog -g2012 -Wall -o sim.vvp tb_alu_4bit.v alu_4bit.v && vvp sim.vvp

In [ ]:
show_waves('dump.vcd')

## Exercise 2: Debounce Module Testbench (25 min)

**SLOs: 6.2, 6.6**

Write `tb_debounce_thorough.v` with four test scenarios:
1. **Clean press:** Input settles low, verify `o_clean` transitions after threshold
2. **Bounce rejection:** 6+ toggles within threshold window, verify single transition
3. **Glitch rejection:** Input low for less than threshold, verify no transition
4. **Clean release:** After stable press, verify release transition

---

#### 📝 Exercise 2 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile debounce.v
// =============================================================================
// debounce.v — Counter-Based Button Debouncer (Solution)
// Day 5, Exercise 1
// =============================================================================

module debounce #(
    parameter CLKS_TO_STABLE = 250_000
)(
    input  wire i_clk,
    input  wire i_bouncy,
    output reg  o_clean
);

    reg [$clog2(CLKS_TO_STABLE)-1:0] r_count;
    reg r_sync_0, r_sync_1;

    always @(posedge i_clk) begin
        // 2-FF synchronizer
        r_sync_0 <= i_bouncy;
        r_sync_1 <= r_sync_0;

        // Debounce logic
        if (r_sync_1 != o_clean) begin
            r_count <= r_count + 1;
            if (r_count == CLKS_TO_STABLE - 1) begin
                o_clean <= r_sync_1;
                r_count <= 0;
            end
        end else begin
            r_count <= 0;
        end
    end

endmodule

In [ ]:
%%writefile tb_debounce_thorough.v
// =============================================================================
// tb_debounce_thorough.v — Thorough Debounce Testbench (Starter)
// Day 6, Exercise 2
// =============================================================================

`timescale 1ns / 1ps

module tb_debounce_thorough;

    reg  clk, bouncy;
    wire clean;

    localparam THRESHOLD = 20;  // short for simulation

    debounce #(.CLKS_TO_STABLE(THRESHOLD)) uut (
        .i_clk(clk), .i_bouncy(bouncy), .o_clean(clean)
    );

    initial clk = 0;
    always #20 clk = ~clk;

    integer test_count = 0, fail_count = 0;

    task wait_clks;
        input integer n;
        begin repeat (n) @(posedge clk); end
    endtask

    initial begin
        $dumpfile("dump.vcd");
        $dumpvars(0, tb_debounce_thorough);
        bouncy = 1;
        wait_clks(10);

        // ---- TODO: Scenario 1 — Clean press ----
        // Set bouncy=0, wait THRESHOLD + sync latency cycles
        // Verify clean transitions to 0

        // ---- TODO: Scenario 2 — Bounce rejection ----
        // First release to 1, then toggle 6+ times within threshold
        // Then settle at 0. Verify clean transitions only once

        // ---- TODO: Scenario 3 — Glitch rejection ----
        // Release bouncy=1, wait for clean=1
        // Then glitch: bouncy=0 for fewer than THRESHOLD clocks, bouncy=1
        // Verify clean never goes to 0

        // ---- TODO: Scenario 4 — Clean release ----
        // Ensure a stable press (clean=0), then release bouncy=1
        // Wait and verify clean returns to 1

        $display("");
        $display("========================================");
        $display("Debounce: %0d / %0d tests PASSED",
                 test_count - fail_count, test_count);
        $display("========================================");
        $finish;
    end

endmodule

In [ ]:
%%writefile Makefile
# Auto-generated Makefile
TOP      = debounce
SRCS     = debounce.v
TB       = tb_debounce_thorough.v
PCF      = ../../go_board.pcf
DEVICE   = hx1k
PACKAGE  = vq100
IVFLAGS  = -g2012 -Wall

sim: $(TB) $(SRCS)
	iverilog $(IVFLAGS) -o sim.vvp $(TB) $(SRCS)
	vvp sim.vvp

wave: sim
	gtkwave *.vcd &

synth: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP) -json $(TOP).json" $(SRCS)

prog: synth
	nextpnr-ice40 --$(DEVICE) --package $(PACKAGE) --pcf $(PCF) --json $(TOP).json --asc $(TOP).asc --freq 25
	icepack $(TOP).asc $(TOP).bin
	iceprog $(TOP).bin

stat: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP); stat" $(SRCS)

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: sim wave synth prog stat clean

In [ ]:
# Compile and simulate
!iverilog -g2012 -Wall -o sim.vvp tb_debounce_thorough.v debounce.v && vvp sim.vvp

In [ ]:
show_waves('dump.vcd')

## Exercise 3: Counter Testbench (20 min)

**SLOs: 6.2, 6.3**

Write `tb_counter.v` for a simple 4-bit counter. Verify:
1. Reset sets count to 0
2. Counter increments each enabled clock
3. Rollover from F to 0
4. Enable=0 holds the count

---

#### 📝 Exercise 3 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile counter_4bit.v
// =============================================================================
// counter_4bit.v — Simple 4-bit Counter (Provided — DUT for Exercise 3)
// Day 6, Exercise 3
// =============================================================================

module counter_4bit (
    input  wire       i_clk,
    input  wire       i_reset,
    input  wire       i_enable,
    output reg  [3:0] o_count,
    output wire       o_zero
);

    always @(posedge i_clk) begin
        if (i_reset)
            o_count <= 4'd0;
        else if (i_enable)
            o_count <= o_count + 4'd1;
    end

    assign o_zero = (o_count == 4'd0);

endmodule

In [ ]:
%%writefile tb_counter.v
// =============================================================================
// tb_counter.v — Counter Testbench (Starter)
// Day 6, Exercise 3
// =============================================================================

`timescale 1ns / 1ps

module tb_counter;

    reg        clk, reset, enable;
    wire [3:0] count;
    wire       zero;

    counter_4bit uut (
        .i_clk(clk), .i_reset(reset), .i_enable(enable),
        .o_count(count), .o_zero(zero)
    );

    initial clk = 0;
    always #20 clk = ~clk;

    integer test_count = 0;
    integer fail_count = 0;

    // ---- TODO: Implement check_count task ----
    // task check_count;
    //   input [3:0] expected;
    //   input       exp_zero;
    //   input [8*32-1:0] label;
    // begin
    //   @(posedge clk); #1;
    //   test_count = test_count + 1;
    //   if (count !== expected || zero !== exp_zero) begin
    //     fail_count = fail_count + 1;
    //     $display("FAIL %0s: expected count=%h zero=%b, got count=%h zero=%b",
    //              label, expected, exp_zero, count, zero);
    //   end else
    //     $display("PASS %0s: count=%h zero=%b", label, count, zero);
    // end
    // endtask

    initial begin
        $dumpfile("dump.vcd");
        $dumpvars(0, tb_counter);

        $display("=== Counter Testbench ===");

        // ---- TODO: Test 1 — Reset ----
        // Apply reset, verify count=0 and zero=1
        reset = 1; enable = 0;
        repeat (3) @(posedge clk);
        // check_count(4'h0, 1'b1, "Reset");

        // ---- TODO: Test 2 — Count up sequence ----
        // Release reset, enable counting, verify 0→1→2→...
        reset = 0; enable = 1;
        // check_count(4'h1, 1'b0, "Count to 1");
        // check_count(4'h2, 1'b0, "Count to 2");

        // ---- TODO: Test 3 — Enable hold ----
        // Disable enable for 5 cycles, verify count holds
        enable = 0;
        repeat (5) @(posedge clk); #1;
        // Verify count hasn't changed

        // ---- TODO: Test 4 — Rollover from F to 0 ----
        // Enable counting, run until rollover
        // Reset and manually count to 4'hF, then one more

        // ---- TODO: Print summary ----
        $display("\n========================================");
        $display("Counter: %0d/%0d tests passed",
                 test_count - fail_count, test_count);
        $display("========================================");
        $finish;
    end

endmodule

In [ ]:
%%writefile Makefile
# Auto-generated Makefile
TOP      = counter_4bit
SRCS     = counter_4bit.v
TB       = tb_counter.v
PCF      = ../../go_board.pcf
DEVICE   = hx1k
PACKAGE  = vq100
IVFLAGS  = -g2012 -Wall

sim: $(TB) $(SRCS)
	iverilog $(IVFLAGS) -o sim.vvp $(TB) $(SRCS)
	vvp sim.vvp

wave: sim
	gtkwave *.vcd &

synth: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP) -json $(TOP).json" $(SRCS)

prog: synth
	nextpnr-ice40 --$(DEVICE) --package $(PACKAGE) --pcf $(PCF) --json $(TOP).json --asc $(TOP).asc --freq 25
	icepack $(TOP).asc $(TOP).bin
	iceprog $(TOP).bin

stat: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP); stat" $(SRCS)

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: sim wave synth prog stat clean

In [ ]:
# Compile and simulate
!iverilog -g2012 -Wall -o sim.vvp tb_counter.v counter_4bit.v && vvp sim.vvp

In [ ]:
show_waves('dump.vcd')

## Exercise 4 (Stretch): File-Driven Testing (15 min)

**SLO: 6.4**

Use `$readmemh` to load test vectors from `hex_vectors.hex`. Verify the hex-to-7seg decoder against file-based expected outputs.

---

#### 📝 Exercise 4 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile hex_to_7seg.v
// =============================================================================
// hex_to_7seg.v — Hex to 7-Segment Decoder (Provided from Day 2)
// =============================================================================

module hex_to_7seg (
    input  wire [3:0] i_hex,
    output reg  [6:0] o_seg    // {a, b, c, d, e, f, g} active-low
);

    always @(*) begin
        case (i_hex)
            4'h0: o_seg = 7'b1111110;
            4'h1: o_seg = 7'b0110000;
            4'h2: o_seg = 7'b1101101;
            4'h3: o_seg = 7'b1111001;
            4'h4: o_seg = 7'b0110011;
            4'h5: o_seg = 7'b1011011;
            4'h6: o_seg = 7'b1011111;
            4'h7: o_seg = 7'b1110000;
            4'h8: o_seg = 7'b1111111;
            4'h9: o_seg = 7'b1111011;
            4'hA: o_seg = 7'b1110111;
            4'hB: o_seg = 7'b0011111;
            4'hC: o_seg = 7'b1001110;
            4'hD: o_seg = 7'b0111101;
            4'hE: o_seg = 7'b1001111;
            4'hF: o_seg = 7'b1000111;
            default: o_seg = 7'b0000000;
        endcase
    end

endmodule

In [ ]:
%%writefile hex_vectors.hex
// hex_vectors.hex — Test vectors for hex_to_7seg
// Format: {input_hex[3:0], expected_seg[6:0]}  (11 bits per vector)
// Segments: a=bit6, b=bit5, c=bit4, d=bit3, e=bit2, f=bit1, g=bit0
// Active-low segments (0 = on, 1 = off)
7E0  // 0: 0111_1110_0 → abcdef on, g off → 0x7E, g=0 → 0x7E0
330  // 1: 0011_0000_0 → bc on → ...
6D0  // 2: 0110_1101_0 → abdeg
790  // 3: 0111_1001_0 → abcdg
330  // 4: 0011_0011_0 → bcfg  (revisit encoding)
5B0  // 5: 0101_1011_0 → acdfg
5F0  // 6: 0101_1111_0 → acdefg
700  // 7: 0111_0000_0 → abc
7F0  // 8: 0111_1111_0 → abcdefg
7B0  // 9: 0111_1011_0 → abcdfg
770  // A: 0111_0111_0 → abcefg
1F0  // b: 0001_1111_0 → cdefg
4E0  // C: 0100_1110_0 → adef
3D0  // d: 0011_1101_0 → bcdeg
4F0  // E: 0100_1111_0 → adefg
470  // F: 0100_0111_0 → aefg

In [ ]:
%%writefile tb_hex_file.v
// =============================================================================
// tb_hex_file.v — File-Driven Hex-to-7seg Testbench (Starter)
// Day 6, Exercise 4 (Stretch)
// =============================================================================
// Uses $readmemh to load test vectors from hex_vectors.hex

`timescale 1ns / 1ps

module tb_hex_file;

    reg  [3:0] hex_in;
    wire [6:0] seg_out;

    hex_to_7seg uut (
        .i_hex(hex_in),
        .o_seg(seg_out)
    );

    // ---- TODO: Storage for test vectors ----
    // reg [10:0] vectors [0:15];  // 11 bits: {4-bit input, 7-bit expected}
    // integer i;
    // integer test_count = 0, fail_count = 0;

    initial begin
        $dumpfile("dump.vcd");
        $dumpvars(0, tb_hex_file);

        $display("=== File-Driven Hex-to-7seg Testbench ===");

        // ---- TODO: Load vectors from file ----
        // $readmemh("hex_vectors.hex", vectors);

        // ---- TODO: Loop through all 16 vectors ----
        // for (i = 0; i < 16; i = i + 1) begin
        //     hex_in = vectors[i][10:7];           // upper 4 bits = input
        //     #10;
        //     test_count = test_count + 1;
        //     if (seg_out !== vectors[i][6:0]) begin
        //         fail_count = fail_count + 1;
        //         $display("FAIL: hex=%h expected seg=%07b got=%07b",
        //                  hex_in, vectors[i][6:0], seg_out);
        //     end else
        //         $display("PASS: hex=%h seg=%07b", hex_in, seg_out);
        // end

        // ---- TODO: Print summary ----

        $finish;
    end

endmodule

In [ ]:
%%writefile Makefile
# Auto-generated Makefile
TOP      = hex_to_7seg
SRCS     = hex_to_7seg.v
TB       = tb_hex_file.v
PCF      = ../../go_board.pcf
DEVICE   = hx1k
PACKAGE  = vq100
IVFLAGS  = -g2012 -Wall

sim: $(TB) $(SRCS)
	iverilog $(IVFLAGS) -o sim.vvp $(TB) $(SRCS)
	vvp sim.vvp

wave: sim
	gtkwave *.vcd &

synth: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP) -json $(TOP).json" $(SRCS)

prog: synth
	nextpnr-ice40 --$(DEVICE) --package $(PACKAGE) --pcf $(PCF) --json $(TOP).json --asc $(TOP).asc --freq 25
	icepack $(TOP).asc $(TOP).bin
	iceprog $(TOP).bin

stat: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP); stat" $(SRCS)

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: sim wave synth prog stat clean

In [ ]:
# Compile and simulate
!iverilog -g2012 -Wall -o sim.vvp tb_hex_file.v hex_to_7seg.v && vvp sim.vvp

In [ ]:
show_waves('dump.vcd')

## Exercise 5 (Stretch): Exhaustive Combinational Test (20 min)

**SLO: 6.2**

Test all 1024 ALU input combinations (4-bit a × 4-bit b × 2-bit opcode). Report: "Tested 1024 combinations, 0 failures."

#### 📝 Exercise 5 — Starter Files

Edit the code below, then run the build cells.

In [ ]:
%%writefile alu_4bit.v
// =============================================================================
// alu_4bit.v — 4-bit ALU (from Day 3, provided for testbench exercises)
// =============================================================================

module alu_4bit (
    input  wire [3:0] i_a,
    input  wire [3:0] i_b,
    input  wire [1:0] i_opcode,
    output reg  [3:0] o_result,
    output reg        o_carry,
    output wire       o_zero
);

    reg [4:0] r_temp;  // 5 bits for carry detection

    always @(*) begin
        r_temp = 5'd0;
        case (i_opcode)
            2'b00: r_temp = {1'b0, i_a} + {1'b0, i_b};  // ADD
            2'b01: r_temp = {1'b0, i_a} - {1'b0, i_b};  // SUB
            2'b10: r_temp = {1'b0, i_a & i_b};           // AND
            2'b11: r_temp = {1'b0, i_a | i_b};           // OR
        endcase
        o_result = r_temp[3:0];
        o_carry  = r_temp[4];
    end

    assign o_zero = (o_result == 4'd0);

endmodule

In [ ]:
%%writefile tb_alu_exhaustive.v
// =============================================================================
// tb_alu_exhaustive.v — Exhaustive ALU Test (Starter)
// Day 6, Exercise 5 (Stretch)
// =============================================================================
// Tests all 1024 input combinations: 16 × 16 × 4 opcodes

`timescale 1ns / 1ps

module tb_alu_exhaustive;

    reg  [3:0] a, b;
    reg  [1:0] opcode;
    wire [3:0] result;
    wire       carry, zero;

    alu_4bit uut (
        .i_a(a), .i_b(b), .i_opcode(opcode),
        .o_result(result), .o_carry(carry), .o_zero(zero)
    );

    integer ia, ib, iop;
    integer test_count = 0, fail_count = 0;
    reg [4:0] expected;

    initial begin
        $dumpfile("dump.vcd");
        $dumpvars(0, tb_alu_exhaustive);

        $display("=== Exhaustive ALU Test (1024 vectors) ===");

        // ---- TODO: Triple-nested loop over all inputs ----
        // for (iop = 0; iop < 4; iop = iop + 1) begin
        //   for (ia = 0; ia < 16; ia = ia + 1) begin
        //     for (ib = 0; ib < 16; ib = ib + 1) begin
        //       a = ia[3:0]; b = ib[3:0]; opcode = iop[1:0];
        //       #10;
        //
        //       // Compute expected result
        //       case (iop[1:0])
        //         2'b00: expected = ia[3:0] + ib[3:0];
        //         2'b01: expected = ia[3:0] - ib[3:0];
        //         2'b10: expected = {1'b0, ia[3:0] & ib[3:0]};
        //         2'b11: expected = {1'b0, ia[3:0] | ib[3:0]};
        //       endcase
        //
        //       test_count = test_count + 1;
        //       if (result !== expected[3:0]) begin
        //         fail_count = fail_count + 1;
        //         $display("FAIL: op=%b a=%h b=%h exp=%h got=%h",
        //                  iop[1:0], ia[3:0], ib[3:0], expected[3:0], result);
        //       end
        //     end
        //   end
        // end

        $display("\n========================================");
        $display("Exhaustive: %0d vectors, %0d failures", test_count, fail_count);
        if (fail_count == 0)
            $display("ALL PASS!");
        $display("========================================");
        $finish;
    end

endmodule

In [ ]:
%%writefile Makefile
# Auto-generated Makefile
TOP      = alu_4bit
SRCS     = alu_4bit.v
TB       = tb_alu_exhaustive.v
PCF      = ../../go_board.pcf
DEVICE   = hx1k
PACKAGE  = vq100
IVFLAGS  = -g2012 -Wall

sim: $(TB) $(SRCS)
	iverilog $(IVFLAGS) -o sim.vvp $(TB) $(SRCS)
	vvp sim.vvp

wave: sim
	gtkwave *.vcd &

synth: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP) -json $(TOP).json" $(SRCS)

prog: synth
	nextpnr-ice40 --$(DEVICE) --package $(PACKAGE) --pcf $(PCF) --json $(TOP).json --asc $(TOP).asc --freq 25
	icepack $(TOP).asc $(TOP).bin
	iceprog $(TOP).bin

stat: $(SRCS)
	yosys -p "synth_ice40 -top $(TOP); stat" $(SRCS)

clean:
	rm -f *.json *.asc *.bin *.vvp *.vcd *.log

.PHONY: sim wave synth prog stat clean

In [ ]:
# Compile and simulate
!iverilog -g2012 -Wall -o sim.vvp tb_alu_exhaustive.v alu_4bit.v && vvp sim.vvp

In [ ]:
show_waves('dump.vcd')